In [ ]:
!pip install -q transformers accelerate torch torchvision pillow tqdm pandas

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))

Python: 3.12.12
Torch: 2.8.0+cu126
CUDA available: True
CUDA device name: Tesla T4


In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_path = "/content/original_images"

# Giải nén
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"✅ Extracted to: {extract_path}")
print("📁 Files:", os.listdir(extract_path)[:5])

In [ ]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
from tqdm import tqdm
import os
import re
import pandas as pd

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")
!ls -R /content/original_images/

✅ Device: cuda
/content/original_images/:
original_images

/content/original_images/original_images:
100.jpg  188.jpg  270.jpg  354.jpg  437.jpg  51.jpg   602.jpg  686.jpg	769.jpg
101.jpg  189.jpg  271.jpg  355.jpg  438.jpg  520.jpg  603.jpg  687.jpg	76.jpg
102.jpg  18.jpg   272.jpg  356.jpg  439.jpg  521.jpg  604.jpg  688.jpg	770.jpg
103.jpg  190.jpg  273.jpg  357.jpg  43.jpg   522.jpg  605.jpg  689.jpg	771.jpg
104.jpg  191.jpg  274.jpg  358.jpg  440.jpg  523.jpg  606.jpg  68.jpg	772.jpg
105.jpg  192.jpg  275.jpg  359.jpg  441.jpg  524.jpg  607.jpg  690.jpg	773.jpg
106.jpg  193.jpg  276.jpg  35.jpg   442.jpg  525.jpg  608.jpg  691.jpg	774.jpg
107.jpg  194.jpg  277.jpg  360.jpg  443.jpg  526.jpg  609.jpg  692.jpg	775.jpg
108.jpg  195.jpg  278.jpg  361.jpg  444.jpg  527.jpg  60.jpg   693.jpg	776.jpg
109.jpg  196.jpg  279.jpg  362.jpg  445.jpg  528.jpg  610.jpg  694.jpg	777.jpg
10.jpg	 197.jpg  27.jpg   363.jpg  446.jpg  529.jpg  611.jpg  695.jpg	778.jpg
110.jpg  198.jpg  280.jpg  364.jp

In [ ]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/5.81G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
folder_path = "/content/original_images/original_images"
records = []

for filename in tqdm(sorted(os.listdir(folder_path))):
    if filename.lower().endswith((".jpg", ".png", ".jpeg", ".bmp", ".webp")):
        match = re.match(r"(\d+)", filename)
        if not match:
            continue
        post_id = int(match.group(1))
        image_path = os.path.join(folder_path, filename)
        image = Image.open(image_path).convert("RGB")

        prompt = "Describe this fashion item in detail, including its color, style, and any notable features."
        inputs = processor(image, prompt, return_tensors="pt").to(device, torch.float16 if device == "cuda" else torch.float32)
        out = model.generate(**inputs, max_new_tokens=100)
        caption = processor.decode(out[0], skip_special_tokens=True)

        records.append({"post_id": post_id, "description": caption})
        print(f"{filename} → {caption}")

df = pd.DataFrame(records).sort_values("post_id")
df.head()

  0%|          | 1/822 [00:00<10:01,  1.36it/s]

1.jpg → a white dress with a white blouse and white pants


  0%|          | 2/822 [00:01<12:05,  1.13it/s]

10.jpg → a bag is a bag that is used to carry a bag or other item


  0%|          | 3/822 [00:02<11:24,  1.20it/s]

...

100%|█████████▉| 821/822 [16:04<00:01,  1.00s/it]

98.jpg → a pair of jeans with a tee and a sweater


100%|██████████| 822/822 [16:05<00:00,  1.17s/it]

99.jpg → a pair of jeans shorts


,post_id,description
0,1,a white dress with a white blouse and white pants
107,2,a dress is a garment that is worn by a woman
218,3,a dress is a dress that is worn by a woman
328,4,a pair of jeans with a chanel logo on the side
439,5,a dress is a dress that is worn by a woman
